# GIF Link Replacements

Create a loop which:
- Feeds a headline into an LLM and asks for GIPHY search terms
- Feeds the search terms into GIPHY API
- Returns potential replacement GIFs

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel
import requests

data_path = "../data"

load_dotenv(override=True)
openAI_api_key = os.getenv('OPENAI_API_KEY')
giphy_api_key = os.getenv("GIPHY_API_KEY")

MODEL = "gpt-5.4-mini"

if not openAI_api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not openAI_api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif openAI_api_key.strip() != openAI_api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

client = OpenAI()

API key found and looks good so far!


In [ ]:
class GIFSearch(BaseModel):
    """Structured returned search terms for GIFs"""
    search_terms: list[str]

system_prompt = "Given a headline, you should return brief search terms for GIFs that could be used as a humorous representation of a notable aspect of the headline. Each search term should be a maximum of three words, but ideally only one or two words. Return five search terms only."

user_prompt_prefix = "Here is the headline, please provide five search terms for humorous GIFs that represent a notable aspect of the headline:"

headline = "MeWork: Meet the Atlanta entrepreneur renting out desks to stranded work-from-homers"



In [ ]:
response = client.responses.parse(
    model=MODEL, 
    instructions=system_prompt,
    input=user_prompt_prefix + headline,
    text_format=GIFSearch
)

result = response.output_parsed

In [65]:
print(result)

search_terms=['rental desk', 'home office', 'stranded worker', 'startup hustle', 'desk for rent']


In [ ]:
with open(f"{data_path}/known_issues_outstanding.json", "r") as f:
    issues = json.load(f)

GIF_replacements = []

for item in issues:
    if issues[item]['headline_issues']:
        for headline_issue in issues[item]['headline_issues']:
            if headline_issue['description'] == "Dead GIF link, replacement required":
                GIF_replacements.append(headline_issue['headline'])

# I am manually adding in the other two replacements for articles with no GIF as it's not worth trying to automate with my current data structure
other_replacements = ["New census question will ask whether Americans can identify Ukraine on a map",
                      "MeWork: Meet the Atlanta entrepreneur renting out desks to stranded work-from-homers"]

GIF_replacements.extend(other_replacements)

GIF_replacements

['Swans with brains 1/300 the size of orangutans’ pass the same IQ test',
 'GM CEO Mary Barra agrees to debate battery tech with Elon Musk on Clubhouse',
 'Slack’s CEO thinks a dating app for coworkers could be in the company’s future',
 'A Chicago museum kicked out a woman it thought was a soup-throwing climate protester. She was actually the museum president',
 'Lizards have strongest kneecaps in animal kingdom, but easily ‘lose’ them',
 'New census question will ask whether Americans can identify Ukraine on a map',
 'MeWork: Meet the Atlanta entrepreneur renting out desks to stranded work-from-homers']

In [ ]:
headline_searches = []

for headline in GIF_replacements:
    response = client.responses.parse(
        model=MODEL, 
        instructions=system_prompt,
        input=user_prompt_prefix + headline,
        text_format=GIFSearch
    )
    headline_searches.append(response.output_parsed)

headline_searches

[GIFSearch(search_terms=['smart swan', 'tiny brain', 'iq test', 'surprised bird', 'genius animal']),
 GIFSearch(search_terms=['debate showdown', 'battery battle', 'Clubhouse chat', 'Face off', 'talking heads']),
 GIFSearch(search_terms=['office romance', 'workplace dating', 'coworker crush', 'company matchmaking', 'CEO idea']),
 GIFSearch(search_terms=['wrong person', 'mix up', 'facepalm', 'oops', 'plot twist']),
 GIFSearch(search_terms=['strong knees', 'pop off', 'oops fell', 'slip away', 'reptile fail']),
 GIFSearch(search_terms=['map fail', 'geography quiz', "can't find Ukraine", 'confused pointing', 'wrong country']),
 GIFSearch(search_terms=['home office', 'desk rental', 'work from home', 'business hustle', 'stranded worker'])]

In [82]:

headline_searches[0].search_terms

['smart swan', 'tiny brain', 'iq test', 'surprised bird', 'genius animal']

In [76]:
def gif_search(search_term):

    gif_links = []

    response = requests.get(
        "https://api.giphy.com/v1/gifs/search",
        params={
            "api_key": giphy_api_key,
            "q": search_term,
            "limit": 5
        }
    )

    for item in response.json()['data']:
        gif_links.append(item['url'])

    return gif_links


In [ ]:
data['data'][0]['url']

In [ ]:
for headline in headline_searches:
    for search in headline.search_terms:
        print(f"{search}, {gif_search(search)}")


smart swan, ['https://giphy.com/gifs/fun-movie-cartoon-7ax1EvwB7yizOniIpm', 'https://giphy.com/gifs/augenblickstudios-boy-swan-Xk5XJm3aMgUxHC3K2T', 'https://giphy.com/gifs/racingwives-samantha-busch-racing-wives-ashley-QWA9kZwHqR4NSop0DX', 'https://giphy.com/gifs/swan-BNcYsVbZVlYHu', 'https://giphy.com/gifs/culture--think-hmm-d3mlE7uhX8KFgEmY']
tiny brain, ['https://giphy.com/gifs/stupid-low-iq-air-brain-zAuaShP97y5IEaFwtJ', 'https://giphy.com/gifs/xray-hardpunjabigifs-brain-2QefkxRsY3jNsa2Zj1', 'https://giphy.com/gifs/the-brain-xray-fool-foolish-wheY7s3MLz96DjzNaV', 'https://giphy.com/gifs/Friends-season-6-friends-tv-the-one-with-ross-denial-VEsfbW0pBu145PPhOi', 'https://giphy.com/gifs/brain-brainless-qwertyy-Ss0uxnoeWMdIOl64nG']
iq test, ['https://giphy.com/gifs/EQnPpWsiFhA8DkUrhX', 'https://giphy.com/gifs/nbc-jane-lynch-weakest-link-you-are-the-ax1b1FMLeI5EdmYeSf', 'https://giphy.com/gifs/smart-idiot-unicornification-9T11Z8O7z7MeDrZKCJ', 'https://giphy.com/gifs/correct-false-brokuza